# Epi Info AI algorithm validation lab — V0.1

This notebook demonstrates one JupyterLite **Python kernel** using two computational engines: the deployed Rust/WebAssembly kernel and an independent SciPy reference. It uses synthetic data only.

> **Validation state:** `epi.table2x2` is a candidate operation. This transparent demonstration is not, by itself, statistical approval. GitLab CI and the reviewed evidence pack remain authoritative.


## 1. Load the versioned fixture

The same JSON fixture is copied into the production artifact by the normal application build.


In [ ]:
import hashlib
import math
import platform
import sys

from pyodide.http import pyfetch

FIXTURE_URL = "/validation-fixtures/table2x2-baseline.json"
fixture_response = await pyfetch(FIXTURE_URL)
fixture_response.raise_for_status()
fixture = await fixture_response.json()
fixture


## 2. Load and call the deployed Rust/WASM engine

Rust is compiled ahead of time in GitLab CI. It is loaded as a library through Pyodide's JavaScript bridge; it is not a second notebook kernel.


In [ ]:
from js import WebAssembly, fetch

WASM_URL = "/epi2x2.wasm"
wasm_response = await fetch(WASM_URL)
if not wasm_response.ok:
    raise RuntimeError(f"Unable to load {WASM_URL}: HTTP {wasm_response.status}")
wasm_buffer = await wasm_response.arrayBuffer()
wasm_module = await WebAssembly.instantiate(wasm_buffer)
rust = wasm_module.instance.exports

values = fixture["input"]
a = values["exposedCases"]
b = values["exposedNonCases"]
c = values["unexposedCases"]
d = values["unexposedNonCases"]

rust_risk_ratio = float(rust.risk_ratio(a, b, c, d))
rust_risk_ratio


## 3. Calculate the independent Python reference

JupyterLite loads SciPy into the Pyodide kernel on first import.


In [ ]:
import scipy
from scipy.stats.contingency import relative_risk

python_result = relative_risk(
    exposed_cases=a,
    exposed_total=a + b,
    control_cases=c,
    control_total=c + d,
)
python_risk_ratio = float(python_result.relative_risk)
python_risk_ratio


## 4. Compare Rust/WASM, SciPy, and the reviewed fixture


In [ ]:
import pandas as pd

expected = float(fixture["expected"]["riskRatio"])
tolerance = float(fixture["absoluteTolerance"])

comparison = pd.DataFrame([
    {
        "comparison": "Rust/WASM vs fixture",
        "observed": rust_risk_ratio,
        "reference": expected,
        "absolute_difference": abs(rust_risk_ratio - expected),
        "tolerance": tolerance,
        "passed": math.isclose(rust_risk_ratio, expected, rel_tol=0, abs_tol=tolerance),
    },
    {
        "comparison": "Rust/WASM vs SciPy",
        "observed": rust_risk_ratio,
        "reference": python_risk_ratio,
        "absolute_difference": abs(rust_risk_ratio - python_risk_ratio),
        "tolerance": tolerance,
        "passed": math.isclose(rust_risk_ratio, python_risk_ratio, rel_tol=0, abs_tol=tolerance),
    },
])

assert comparison["passed"].all(), comparison
comparison


## 5. Capture provenance

A validation record must identify the method, fixture, engines, and exact WASM artifact.


In [ ]:
wasm_hash_response = await pyfetch(WASM_URL)
wasm_hash_response.raise_for_status()
wasm_bytes = await wasm_hash_response.bytes()

provenance = {
    "operation": fixture["contract"]["operation"],
    "result_schema_version": fixture["contract"]["schemaVersion"],
    "engine_id": fixture["contract"]["engineId"],
    "engine_version": fixture["contract"]["engineVersion"],
    "wasm_sha256": hashlib.sha256(wasm_bytes).hexdigest(),
    "python": sys.version.split()[0],
    "python_platform": platform.platform(),
    "scipy": scipy.__version__,
    "absolute_tolerance": tolerance,
    "all_comparisons_passed": bool(comparison["passed"].all()),
}
provenance


## Next validation increment

V0.2 should import the legacy 100-case `TwoBy2Stats.csv` corpus, classify every output by method and tolerance, and render all discrepancies without replacing the GitLab validation gates.
